In [ ]:
##### IMPORTANT - to run this script outside of the OFH-GOCT-v14 project, you will need to copy over the /users and /data folders to your
##### project of choice

## Configuration - choose source GRS and OR column here

# source can be set to:
#   - "Conti" for Conti et al. 2021 GRS
#   - "Wang" for Wang et al. 2023 GRS

# weightings can be set to:
#   - "Multiethnic" for the multiethnic ORs
#   - "EUR" for the European genetic similarity ORs
#   - "AFR" for the African genetic similarity ORs
#   - "EAS" for the East Asian genetic similarity ORs
#   - "HIS" for the Hispanic genetic similarity ORs

source <- "Conti" 
weightings <- "Multiethnic"
output_dir <- "/landingzone"  # Specify the output directory for the GRS results

In [ ]:
# This codeblock is used to download the supplementary tables from the Conti et al. (2021) and Wang et al. (2023) papers, 
# and convert them into varlists

system("bash -c 'unset HTTP_PROXY HTTPS_PROXY http_proxy https_proxy; dx download project-J8PJg1V25y16J0Y3F05pPJBj:/users/luke_p/scripts_genetics/polygenic_score.sh'")  # Luke's polygenic score script

#system("bash -c 'unset HTTP_PROXY HTTPS_PROXY http_proxy https_proxy; dx download project-J8PJg1V25y16J0Y3F05pPJBj:/users/callum/Wang2023supplementarytables.xlsx'") 
system("bash -c 'unset HTTP_PROXY HTTPS_PROXY http_proxy https_proxy; dx download project-J8PJg1V25y16J0Y3F05pPJBj:/users/callum/Wang2023supplementarytable4.csv'")

#system("bash -c 'unset HTTP_PROXY HTTPS_PROXY http_proxy https_proxy; dx download project-J8PJg1V25y16J0Y3F05pPJBj:/users/callum/Conti2021supplementarytables.xlsx'") 
system("bash -c 'unset HTTP_PROXY HTTPS_PROXY http_proxy https_proxy; dx download project-J8PJg1V25y16J0Y3F05pPJBj:/users/callum/Conti2021supplementarytable4.csv'")

Wang2023supplementarytable4 <- read.csv("Wang2023supplementarytable4.csv") 
Conti2021supplementarytable4 <- read.csv("Conti2021supplementarytable4.csv")

## let's see how the duplicate column naming affected the formatting of the data frame column names

write.csv(Wang2023supplementarytable4, "Wang2023supplementarytable4_test.csv", row.names = FALSE)
write.csv(Conti2021supplementarytable4, "Conti2021supplementarytable4_test.csv", row.names = FALSE)

In [ ]:
## Now let's rename columns based on original .xlsx spreadsheet

library(dplyr)

if (source == "Wang") {
  raw_data <- Wang2023supplementarytable4 %>%
    dplyr::rename(
      rsid = rsID,
      chr = Chromosome,
      pos_GRCh37 = Position..GRCh37.,
      effect_allele = Risk.Allele,
      other_allele = Reference.Allele,
      Conti = Conti.et.al...269.SNPs,
      Wang = Wang.et.al...451.SNPs,
      EUR_Rsquared = European,
      AFR_Rsquared = African,
      EAS_Rsquared = Asian,          
      HIS_Rsquared = Hispanic,       
      OR_Multiethnic_Marginal = OR,
      CI95_Multiethnic_Marginal = X95.CI,
      Pvalue_Multiethnic_Marginal = P.value,
      OR_Multiethnic_Conditional = OR.1,
      CI95_Multiethnic_Conditional = X95.CI.1,
      Pvalue_Multiethnic_Conditional = P.value.1,
      RAF_EUR = RAF,
      OR_EUR = OR.2,
      CI95_EUR = X95.CI.2,
      Pvalue_EUR = P.value.2,
      RAF_AFR = RAF.1,
      OR_AFR = OR.3,
      CI95_AFR = X95.CI.3,
      Pvalue_AFR = P.value.3,
      RAF_EAS = RAF.2,
      OR_EAS = OR.4,
      CI95_EAS = X95.CI.4,
      Pvalue_EAS = P.value.4,
      RAF_HIS = RAF.3,
      OR_HIS = OR.5,
      CI95_HIS = X95.CI.5,
      Pvalue_HIS = P.value.5
    ) %>% dplyr::filter(!is.na(pos_GRCh37)) ## Removes bottom four rows which have NA or blank for pretty much every column
 } else if (source == "Conti") {
    raw_data <- Conti2021supplementarytable4 %>%
        dplyr::rename(
        rsid = rs.,
        chr = Chromosome,
        pos_GRCh37 = Position,
        effect_allele = Risk.Allele,
        other_allele = Reference.Allele,
        EUR_raf = European,
        AFR_raf = African,
        EAS_raf = East.Asian,          
        HIS_raf = Hispanic,       
        OR_Multiethnic = Multiethnic.Analysis,
        CI95_Multiethnic = X,
        Pvalue_Multiethnic = X.1,
        OR_EUR = European.1,
        CI95_EUR = X.2,
        Pvalue_EUR = X.3,
        OR_AFR = African.1,
        CI95_AFR = X.4,
        Pvalue_AFR = X.5,
        OR_EAS = East.Asian.1,
        CI95_EAS = X.6,
        Pvalue_EAS = X.7,
        OR_HIS = Hispanic.1,
        CI95_HIS = X.8,
        Pvalue_HIS = X.9,
        Blank = X.10,
        OR_UKB_EUR = OR..95..CI...P.value,
        CI95_UKB_EUR = X.11,
        Pvalue_UKB_EUR = X.12,
        UKB_EUR_raf = Risk.Allele.Frequency,
        UKB_EUR_info = Info.Score
    ) %>% dplyr::filter(!is.na(pos_GRCh37)) ## Removes bottom four rows which have NA or blank for pretty much every column
} else {
  stop("Invalid source. Choose 'Conti', or 'Wang'.")
}

## Write to csv to check all columns were renamed correctly

write.csv(raw_data, "raw_data.csv", row.names = FALSE)

In [ ]:
## Convert positions from GRCh37 to GRCh38

raw_data_h38 <- raw_data %>%
    dplyr::mutate(pos_GRCh38 = dplyr::case_when(
      as.character(pos_GRCh37) %in% c("5743196") ~ "5683136",  # rs7542260
      as.character(pos_GRCh37) %in% c("10564675") ~ "10504618",  # rs2847344
      as.character(pos_GRCh37) %in% c("10566272") ~ "10506215",  # rs616402
      as.character(pos_GRCh37) %in% c("16376831") ~ "16050336",  # rs10803412
      as.character(pos_GRCh37) %in% c("16185635") ~ "15859140",  # rs34623941
      as.character(pos_GRCh37) %in% c("24716207") ~ "24389717",  # rs509526
      as.character(pos_GRCh37) %in% c("27679797") ~ "27353306",  # rs3813795
      as.character(pos_GRCh37) %in% c("31381106") ~ "30908259",  # rs12118566
      as.character(pos_GRCh37) %in% c("46251655") ~ "45785983",  # rs544780844
      as.character(pos_GRCh37) %in% c("46425500") ~ "45959828",  # rs4420029
      as.character(pos_GRCh37) %in% c("51464645") ~ "50998973",  # rs6656241
      as.character(pos_GRCh37) %in% c("54831969") ~ "54366296",  # rs11206341
      as.character(pos_GRCh37) %in% c("83120822") ~ "82655139",  # rs365222
      as.character(pos_GRCh37) %in% c("84881095") ~ "84415412",  # rs3116433
      as.character(pos_GRCh37) %in% c("88210715") ~ "87745032",  # rs56391074
      as.character(pos_GRCh37) %in% c("88199778") ~ "87734095",  # rs12131120
      as.character(pos_GRCh37) %in% c("88435308") ~ "87969625",  # rs1413528
      as.character(pos_GRCh37) %in% c("93465613") ~ "93000056",  # rs79239530
      as.character(pos_GRCh37) %in% c("93312203") ~ "92846646",  # rs141153466
      as.character(pos_GRCh37) %in% c("111231499") ~ "110688877",  # rs2797527
      as.character(pos_GRCh37) %in% c("150529322") ~ "150556846",  # rs34379037
      as.character(pos_GRCh37) %in% c("150609429") ~ "150636953",  # rs116173394
      as.character(pos_GRCh37) %in% c("150772613") ~ "150800137",  # rs1811698
      as.character(pos_GRCh37) %in% c("150954671") ~ "150982195",  # rs607518
      as.character(pos_GRCh37) %in% c("153923276") ~ "153950800",  # rs10127983
      as.character(pos_GRCh37) %in% c("153937746") ~ "153965270",  # rs11264734
      as.character(pos_GRCh37) %in% c("154980351") ~ "155007875",  # rs56103503
      as.character(pos_GRCh37) %in% c("155118588") ~ "155146112",  # rs147847496
      as.character(pos_GRCh37) %in% c("155024095") ~ "155051619",  # rs146564277
      as.character(pos_GRCh37) %in% c("155690186") ~ "155720395",  # rs184104770
      as.character(pos_GRCh37) %in% c("156083468") ~ "156113677",  # rs2485662
      as.character(pos_GRCh37) %in% c("157119915") ~ "157150123",  # rs80237341
      as.character(pos_GRCh37) %in% c("157164733") ~ "157194943",  # rs36042807
      as.character(pos_GRCh37) %in% c("163295678") ~ "163325888",  # rs6660538
      as.character(pos_GRCh37) %in% c("163320568") ~ "163350778",  # rs11388275
      as.character(pos_GRCh37) %in% c("167135941") ~ "167166704",  # rs4075646
      as.character(pos_GRCh37) %in% c("167179419") ~ "167210182",  # rs72691600
      as.character(pos_GRCh37) %in% c("179897070") ~ "179927935",  # rs507603
      as.character(pos_GRCh37) %in% c("179894880") ~ "179925745",  # rs555526
      as.character(pos_GRCh37) %in% c("183032447") ~ "183063312",  # rs56677963
      as.character(pos_GRCh37) %in% c("183106623") ~ "183137488",  # rs3835740
      as.character(pos_GRCh37) %in% c("184182110") ~ "184212976",  # rs997343
      as.character(pos_GRCh37) %in% c("199875828") ~ "199906700",  # rs41333244
      as.character(pos_GRCh37) %in% c("204030362") ~ "204061234",  # rs138638958
      as.character(pos_GRCh37) %in% c("204029211") ~ "204060083",  # rs2886543
      as.character(pos_GRCh37) %in% c("204518842") ~ "204549714",  # rs4245739
      as.character(pos_GRCh37) %in% c("204480742") ~ "204511614",  # rs2926533
      as.character(pos_GRCh37) %in% c("205739266") ~ "205770138",  # rs708723
      as.character(pos_GRCh37) %in% c("233512573") ~ "233376827",  # rs1294247
      as.character(pos_GRCh37) %in% c("8598444") ~ "8458314",  # rs11686272
      as.character(pos_GRCh37) %in% c("8597823") ~ "8457693",  # rs10188360
      as.character(pos_GRCh37) %in% c("10094526") ~ "9954397",  # rs73913932
      as.character(pos_GRCh37) %in% c("10781975") ~ "10641849",  # rs1990613
      as.character(pos_GRCh37) %in% c("16016503") ~ "15876379",  # rs7602028
      as.character(pos_GRCh37) %in% c("16027190") ~ "15887066",  # rs4668475
      as.character(pos_GRCh37) %in% c("20878105") ~ "20678345",  # rs9306894
      as.character(pos_GRCh37) %in% c("20878153") ~ "20678393",  # rs9306895
      as.character(pos_GRCh37) %in% c("37885400") ~ "37658257",  # rs6722589
      as.character(pos_GRCh37) %in% c("43064555") ~ "42837415",  # rs6738169
      as.character(pos_GRCh37) %in% c("43073548") ~ "42846408",  # rs4953671
      as.character(pos_GRCh37) %in% c("43637998") ~ "43410859",  # rs7591218
      as.character(pos_GRCh37) %in% c("43551893") ~ "43324754",  # rs73923570
      as.character(pos_GRCh37) %in% c("43851282") ~ "43624143",  # rs28514770
      as.character(pos_GRCh37) %in% c("58361134") ~ "58133999",  # rs12999901
      as.character(pos_GRCh37) %in% c("62752975") ~ "62525840",  # rs11125927
      as.character(pos_GRCh37) %in% c("63277843") ~ "63050708",  # rs58235267
      as.character(pos_GRCh37) %in% c("63938756") ~ "63711622",  # rs139283528
      as.character(pos_GRCh37) %in% c("66652885") ~ "66425753",  # rs74702681
      as.character(pos_GRCh37) %in% c("68482327") ~ "68255195",  # rs2902065
      as.character(pos_GRCh37) %in% c("85767735") ~ "85540612",  # rs2028900
      as.character(pos_GRCh37) %in% c("85769711") ~ "85542588",  # rs1078004
      as.character(pos_GRCh37) %in% c("111861993") ~ "111104416",  # rs2165108
      as.character(pos_GRCh37) %in% c("111904370") ~ "111146793",  # rs4848397
      as.character(pos_GRCh37) %in% c("111893096") ~ "111135519",  # rs11691517
      as.character(pos_GRCh37) %in% c("111903527") ~ "111145950",  # rs3761705
      as.character(pos_GRCh37) %in% c("113399795") ~ "112642218",  # rs4849090
      as.character(pos_GRCh37) %in% c("121103598") ~ "120346022",  # rs111595856
      as.character(pos_GRCh37) %in% c("121103903") ~ "120346327",  # rs11900747
      as.character(pos_GRCh37) %in% c("121239360") ~ "120481784",  # rs4848599
      as.character(pos_GRCh37) %in% c("121373466") ~ "120615890",  # rs10206072
      as.character(pos_GRCh37) %in% c("121382140") ~ "120624564",  # rs6541731
      as.character(pos_GRCh37) %in% c("134535495") ~ "133777924",  # rs1037137
      as.character(pos_GRCh37) %in% c("148461293") ~ "147703724",  # rs79120240
      as.character(pos_GRCh37) %in% c("148548761") ~ "147791192",  # rs11315085
      as.character(pos_GRCh37) %in% c("152247142") ~ "151390628",  # rs79256286
      as.character(pos_GRCh37) %in% c("169012955") ~ "168156445",  # rs16854905
      as.character(pos_GRCh37) %in% c("169035673") ~ "168179163",  # rs11892008
      as.character(pos_GRCh37) %in% c("173319930") ~ "172455202",  # rs77167534
      as.character(pos_GRCh37) %in% c("173313453") ~ "172448725",  # rs1574259
      as.character(pos_GRCh37) %in% c("174234547") ~ "173369819",  # rs34925593
      as.character(pos_GRCh37) %in% c("174852271") ~ "173987543",  # rs7578920
      as.character(pos_GRCh37) %in% c("181276217") ~ "180411490",  # rs67228975
      as.character(pos_GRCh37) %in% c("202126615") ~ "201261892",  # rs1861270
      as.character(pos_GRCh37) %in% c("208118301") ~ "207253577",  # rs12621900
      as.character(pos_GRCh37) %in% c("208031471") ~ "207166747",  # rs57314858
      as.character(pos_GRCh37) %in% c("213949073") ~ "213084349",  # rs6733869
      as.character(pos_GRCh37) %in% c("233520254") ~ "232655544",  # rs13395911
      as.character(pos_GRCh37) %in% c("238411293") ~ "237502650",  # rs74001374
      as.character(pos_GRCh37) %in% c("238443226") ~ "237534583",  # rs2292884
      as.character(pos_GRCh37) %in% c("238395479") ~ "237486836",  # rs58057291
      as.character(pos_GRCh37) %in% c("242141719") ~ "241202304",  # rs2074840
      as.character(pos_GRCh37) %in% c("242123707") ~ "241184292",  # rs2097976
      as.character(pos_GRCh37) %in% c("242135265") ~ "241195850",  # rs77559646
      as.character(pos_GRCh37) %in% c("242139600") ~ "241200185",  # rs77482050
      as.character(pos_GRCh37) %in% c("242157241") ~ "241217826",  # rs76832527
      as.character(pos_GRCh37) %in% c("242163365") ~ "241223950",  # rs60985508
      as.character(pos_GRCh37) %in% c("18738940") ~ "18697448",  # rs6550597
      as.character(pos_GRCh37) %in% c("18258861") ~ "18217369",  # rs4688916
      as.character(pos_GRCh37) %in% c("23153062") ~ "23111571",  # rs7618603
      as.character(pos_GRCh37) %in% c("23155056") ~ "23113565",  # rs9815122
      as.character(pos_GRCh37) %in% c("49621718") ~ "49584285",  # rs34680713
      as.character(pos_GRCh37) %in% c("49796391") ~ "49758958",  # rs140813356
      as.character(pos_GRCh37) %in% c("64920816") ~ "64935141",  # rs73121401
      as.character(pos_GRCh37) %in% c("70796696") ~ "70747545",  # rs13091518
      as.character(pos_GRCh37) %in% c("71238369") ~ "71189218",  # rs6147879
      as.character(pos_GRCh37) %in% c("86991598") ~ "86942448",  # rs77973813
      as.character(pos_GRCh37) %in% c("87144017") ~ "87094867",  # rs143745027
      as.character(pos_GRCh37) %in% c("87154582") ~ "87105432",  # rs78267036
      as.character(pos_GRCh37) %in% c("87175984") ~ "87126834",  # rs7628934
      as.character(pos_GRCh37) %in% c("87221370") ~ "87172220",  # rs4527381
      as.character(pos_GRCh37) %in% c("87399362") ~ "87350212",  # rs114810266
      as.character(pos_GRCh37) %in% c("87199101") ~ "87149951",  # rs77464751
      as.character(pos_GRCh37) %in% c("106962521") ~ "107243674",  # rs1283104
      as.character(pos_GRCh37) %in% c("106961285") ~ "107242438",  # rs1283108
      as.character(pos_GRCh37) %in% c("107193337") ~ "107474490",  # rs151038334
      as.character(pos_GRCh37) %in% c("113300183") ~ "113581336",  # rs2271494
      as.character(pos_GRCh37) %in% c("114732510") ~ "115013663",  # rs72960383
      as.character(pos_GRCh37) %in% c("114696443") ~ "114977596",  # rs58418703
      as.character(pos_GRCh37) %in% c("127022826") ~ "127303983",  # rs184923198
      as.character(pos_GRCh37) %in% c("127898501") ~ "128179658",  # rs2811476
      as.character(pos_GRCh37) %in% c("128213994") ~ "128495151",  # rs35006112
      as.character(pos_GRCh37) %in% c("128212730") ~ "128493887",  # rs4857898
      as.character(pos_GRCh37) %in% c("131984671") ~ "132265827",  # rs61792779
      as.character(pos_GRCh37) %in% c("136671504") ~ "136952662",  # rs835648
      as.character(pos_GRCh37) %in% c("137562823") ~ "137843981",  # rs1457063
      as.character(pos_GRCh37) %in% c("141147414") ~ "141428572",  # rs7650602
      as.character(pos_GRCh37) %in% c("141125186") ~ "141406344",  # rs1344674
      as.character(pos_GRCh37) %in% c("150466940") ~ "150749153",  # rs1879422
      as.character(pos_GRCh37) %in% c("152011745") ~ "152293956",  # rs145053401
      as.character(pos_GRCh37) %in% c("151993952") ~ "152276163",  # rs1426607
      as.character(pos_GRCh37) %in% c("153659873") ~ "153942084",  # rs55681402
      as.character(pos_GRCh37) %in% c("160278478") ~ "160560690",  # rs1317365
      as.character(pos_GRCh37) %in% c("169482335") ~ "169764547",  # rs2293607
      as.character(pos_GRCh37) %in% c("170074517") ~ "170356729",  # rs78416326
      as.character(pos_GRCh37) %in% c("170083540") ~ "170365752",  # rs66512269
      as.character(pos_GRCh37) %in% c("170100476") ~ "170382688",  # rs6784631
      as.character(pos_GRCh37) %in% c("172381777") ~ "172663987",  # rs144600367
      as.character(pos_GRCh37) %in% c("172379603") ~ "172661813",  # rs9836594
      as.character(pos_GRCh37) %in% c("172226240") ~ "172508450",  # rs3136601
      as.character(pos_GRCh37) %in% c("183511570") ~ "183793782",  # rs263023
      as.character(pos_GRCh37) %in% c("74442349") ~ "73576632",  # rs17804499
      as.character(pos_GRCh37) %in% c("74447930") ~ "73582213",  # rs72649118
      as.character(pos_GRCh37) %in% c("74477135") ~ "73611418",  # rs13142786
      as.character(pos_GRCh37) %in% c("74476924") ~ "73611207",  # rs13131954
      as.character(pos_GRCh37) %in% c("77614640") ~ "76693487",  # rs9993810
      as.character(pos_GRCh37) %in% c("95544718") ~ "94623567",  # rs6853490
      as.character(pos_GRCh37) %in% c("103232168") ~ "102311011",  # rs13103835
      as.character(pos_GRCh37) %in% c("106061534") ~ "105140377",  # rs7679673
      as.character(pos_GRCh37) %in% c("106065308") ~ "105144151",  # rs10007915
      as.character(pos_GRCh37) %in% c("106064754") ~ "105143597",  # rs17035310
      as.character(pos_GRCh37) %in% c("106078344") ~ "105157187",  # rs34316731
      as.character(pos_GRCh37) %in% c("110810499") ~ "109889343",  # rs7689017
      as.character(pos_GRCh37) %in% c("110848652") ~ "109927496",  # rs10025078
      as.character(pos_GRCh37) %in% c("140948835") ~ "140027681",  # rs77821238
      as.character(pos_GRCh37) %in% c("140946341") ~ "140025187",  # rs1402672
      as.character(pos_GRCh37) %in% c("146879237") ~ "145958085",  # rs72725734
      as.character(pos_GRCh37) %in% c("152030340") ~ "151109188",  # rs147762399
      as.character(pos_GRCh37) %in% c("152002915") ~ "151081763",  # rs77063077
      as.character(pos_GRCh37) %in% c("157533294") ~ "156612142",  # rs1521717
      as.character(pos_GRCh37) %in% c("170089333") ~ "169168182",  # rs10021065
      as.character(pos_GRCh37) %in% c("1280028") ~ "1279913",  # rs2242652
      as.character(pos_GRCh37) %in% c("1292118") ~ "1292003",  # rs71595003
      as.character(pos_GRCh37) %in% c("1298733") ~ "1298618",  # rs13190087
      as.character(pos_GRCh37) %in% c("1294086") ~ "1293971",  # rs2736098
      as.character(pos_GRCh37) %in% c("1891174") ~ "1891060",  # rs4975758
      as.character(pos_GRCh37) %in% c("1889346") ~ "1889232",  # rs199577062
      as.character(pos_GRCh37) %in% c("14372362") ~ "14372253",  # rs71599622
      as.character(pos_GRCh37) %in% c("14848997") ~ "14848888",  # rs61169618
      as.character(pos_GRCh37) %in% c("37833419") ~ "37833317",  # rs10941370
      as.character(pos_GRCh37) %in% c("43256189") ~ "43256087",  # rs35211243
      as.character(pos_GRCh37) %in% c("44368506") ~ "44368404",  # rs1482675
      as.character(pos_GRCh37) %in% c("44392142") ~ "44392040",  # rs1482680
      as.character(pos_GRCh37) %in% c("51179883") ~ "51884049",  # rs6895682
      as.character(pos_GRCh37) %in% c("52644390") ~ "53348560",  # rs62357764
      as.character(pos_GRCh37) %in% c("56087910") ~ "56792083",  # rs9292122
      as.character(pos_GRCh37) %in% c("56145068") ~ "56849241",  # rs33329
      as.character(pos_GRCh37) %in% c("65800275") ~ "66504447",  # rs6884284
      as.character(pos_GRCh37) %in% c("66011501") ~ "66715673",  # rs139921834
      as.character(pos_GRCh37) %in% c("73392666") ~ "74096841",  # rs4704108
      as.character(pos_GRCh37) %in% c("133836209") ~ "134500518",  # rs10793821
      as.character(pos_GRCh37) %in% c("133848688") ~ "134512997",  # rs11242218
      as.character(pos_GRCh37) %in% c("159486736") ~ "160059729",  # rs4509063
      as.character(pos_GRCh37) %in% c("169172133") ~ "169745129",  # rs76551843
      as.character(pos_GRCh37) %in% c("169015479") ~ "169588475",  # rs116483731
      as.character(pos_GRCh37) %in% c("171960412") ~ "172533408",  # rs72848121
      as.character(pos_GRCh37) %in% c("172959030") ~ "173532027",  # rs9686557
      as.character(pos_GRCh37) %in% c("176603440") ~ "177176439",  # rs144442485
      as.character(pos_GRCh37) %in% c("177683905") ~ "178256904",  # rs61739424
      as.character(pos_GRCh37) %in% c("177891551") ~ "178464550",  # rs2672843
      as.character(pos_GRCh37) %in% c("177900178") ~ "178473177",  # rs10043416
      as.character(pos_GRCh37) %in% c("177977483") ~ "178550482",  # rs566243
      as.character(pos_GRCh37) %in% c("1670985") ~ "1670751",  # rs2814811
      as.character(pos_GRCh37) %in% c("11217897") ~ "11217664",  # rs2018336
      as.character(pos_GRCh37) %in% c("11219030") ~ "11218797",  # rs4713266
      as.character(pos_GRCh37) %in% c("11289748") ~ "11289515",  # rs4713343
      as.character(pos_GRCh37) %in% c("21330689") ~ "21330458",  # rs6927369
      as.character(pos_GRCh37) %in% c("21330808") ~ "21330577",  # rs9348465
      as.character(pos_GRCh37) %in% c("21471490") ~ "21471259",  # rs4269363
      as.character(pos_GRCh37) %in% c("21462138") ~ "21461907",  # rs9466096
      as.character(pos_GRCh37) %in% c("21878849") ~ "21878618",  # rs12665509
      as.character(pos_GRCh37) %in% c("21874359") ~ "21874128",  # rs2182997
      as.character(pos_GRCh37) %in% c("22381368") ~ "22381139",  # rs9393282
      as.character(pos_GRCh37) %in% c("26649830") ~ "26649602",  # rs35159226
      as.character(pos_GRCh37) %in% c("30216712") ~ "30248935",  # rs375611687
      as.character(pos_GRCh37) %in% c("29922740") ~ "29954963",  # rs16896742
      as.character(pos_GRCh37) %in% c("31229552") ~ "31261775",  # rs2844626
      as.character(pos_GRCh37) %in% c("32652620") ~ "32684843",  # rs9275160
      as.character(pos_GRCh37) %in% c("34793124") ~ "34825347",  # rs9469899
      as.character(pos_GRCh37) %in% c("34655000") ~ "34687223",  # rs2744961
      as.character(pos_GRCh37) %in% c("37138220") ~ "37170444",  # rs262936
      as.character(pos_GRCh37) %in% c("41536587") ~ "41568849",  # rs4714485
      as.character(pos_GRCh37) %in% c("41542417") ~ "41574679",  # rs4714486
      as.character(pos_GRCh37) %in% c("43694598") ~ "43726861",  # rs4711748
      as.character(pos_GRCh37) %in% c("43709785") ~ "43742048",  # rs9472120
      as.character(pos_GRCh37) %in% c("43710015") ~ "43742278",  # rs9472121
      as.character(pos_GRCh37) %in% c("53337681") ~ "53472883",  # rs594260
      as.character(pos_GRCh37) %in% c("71282172") ~ "70572469",  # rs77054685
      as.character(pos_GRCh37) %in% c("72020596") ~ "71310893",  # rs16881098
      as.character(pos_GRCh37) %in% c("76495882") ~ "75786165",  # rs9443189
      as.character(pos_GRCh37) %in% c("76489774") ~ "75780057",  # rs6906615
      as.character(pos_GRCh37) %in% c("80757780") ~ "80048063",  # rs146575246
      as.character(pos_GRCh37) %in% c("107597364") ~ "107276160",  # rs62430067
      as.character(pos_GRCh37) %in% c("109295293") ~ "108974090",  # rs2038542
      as.character(pos_GRCh37) %in% c("109312239") ~ "108991036",  # rs78550764
      as.character(pos_GRCh37) %in% c("117200434") ~ "116879271",  # rs339351
      as.character(pos_GRCh37) %in% c("117208509") ~ "116887346",  # rs339328
      as.character(pos_GRCh37) %in% c("130374461") ~ "130053316",  # rs7740107
      as.character(pos_GRCh37) %in% c("134292717") ~ "133971579",  # rs71681861
      as.character(pos_GRCh37) %in% c("134268822") ~ "133947684",  # rs228867
      as.character(pos_GRCh37) %in% c("134341363") ~ "134020225",  # rs10499188
      as.character(pos_GRCh37) %in% c("139859028") ~ "139537891",  # rs34379299
      as.character(pos_GRCh37) %in% c("153447516") ~ "153126381",  # rs13215045
      as.character(pos_GRCh37) %in% c("160150279") ~ "159729247",  # rs963800
      as.character(pos_GRCh37) %in% c("160149462") ~ "159728430",  # rs57022984
      as.character(pos_GRCh37) %in% c("160581543") ~ "160160511",  # rs4646284
      as.character(pos_GRCh37) %in% c("1928159") ~ "1888523",  # rs4513875
      as.character(pos_GRCh37) %in% c("1965255") ~ "1925619",  # rs12671113
      as.character(pos_GRCh37) %in% c("20414110") ~ "20374487",  # rs11452686
      as.character(pos_GRCh37) %in% c("20444081") ~ "20404458",  # rs6973059
      as.character(pos_GRCh37) %in% c("20999211") ~ "20959592",  # rs9655205
      as.character(pos_GRCh37) %in% c("20996952") ~ "20957333",  # rs10713532
      as.character(pos_GRCh37) %in% c("21812043") ~ "21772425",  # rs35389879
      as.character(pos_GRCh37) %in% c("22896340") ~ "22856721",  # rs5882865
      as.character(pos_GRCh37) %in% c("27564862") ~ "27525243",  # rs6956484
      as.character(pos_GRCh37) %in% c("27560990") ~ "27521371",  # rs940526
      as.character(pos_GRCh37) %in% c("27976563") ~ "27936944",  # rs10486567
      as.character(pos_GRCh37) %in% c("36433721") ~ "36394112",  # rs6964685
      as.character(pos_GRCh37) %in% c("40877473") ~ "40837874",  # rs12701838
      as.character(pos_GRCh37) %in% c("40812079") ~ "40772480",  # rs12701832
      as.character(pos_GRCh37) %in% c("47445641") ~ "47406044",  # rs700755
      as.character(pos_GRCh37) %in% c("47451918") ~ "47412320",  # rs834608
      as.character(pos_GRCh37) %in% c("47510577") ~ "47470979",  # rs11770955
      as.character(pos_GRCh37) %in% c("50834298") ~ "50766601",  # rs5884169
      as.character(pos_GRCh37) %in% c("81428307") ~ "81798991",  # rs1019617
      as.character(pos_GRCh37) %in% c("92577760") ~ "92948446",  # rs6955627
      as.character(pos_GRCh37) %in% c("92551114") ~ "92921800",  # rs141155748
      as.character(pos_GRCh37) %in% c("97688440") ~ "98059128",  # rs4727386
      as.character(pos_GRCh37) %in% c("97816327") ~ "98187015",  # rs6465657
      as.character(pos_GRCh37) %in% c("8498803") ~ "8641293",  # rs870167
      as.character(pos_GRCh37) %in% c("8455634") ~ "8598124",  # rs62494897
      as.character(pos_GRCh37) %in% c("11217455") ~ "11359946",  # rs2572375
      as.character(pos_GRCh37) %in% c("11144199") ~ "11286690",  # rs200170336
      as.character(pos_GRCh37) %in% c("22879734") ~ "23022221",  # rs4871844
      as.character(pos_GRCh37) %in% c("23470785") ~ "23613272",  # rs6557704
      as.character(pos_GRCh37) %in% c("23465916") ~ "23608403",  # rs992530
      as.character(pos_GRCh37) %in% c("23529521") ~ "23672008",  # rs1160267
      as.character(pos_GRCh37) %in% c("25894201") ~ "26036685",  # rs141811748
      as.character(pos_GRCh37) %in% c("26063165") ~ "26205649",  # rs12677206
      as.character(pos_GRCh37) %in% c("38644914") ~ "38787396",  # rs11467
      as.character(pos_GRCh37) %in% c("61664214") ~ "60751655",  # rs876957
      as.character(pos_GRCh37) %in% c("71470935") ~ "70558700",  # rs75110624
      as.character(pos_GRCh37) %in% c("81126774") ~ "80214539",  # rs34368548
      as.character(pos_GRCh37) %in% c("95685147") ~ "94672919",  # rs11786992
      as.character(pos_GRCh37) %in% c("102382456") ~ "101370228",  # rs77244444
      as.character(pos_GRCh37) %in% c("108923107") ~ "107910879",  # rs73700335
      as.character(pos_GRCh37) %in% c("108917591") ~ "107905363",  # rs4735020
      as.character(pos_GRCh37) %in% c("109082646") ~ "108070418",  # rs13253117
      as.character(pos_GRCh37) %in% c("127836925") ~ "126824680",  # rs57588856
      as.character(pos_GRCh37) %in% c("127861121") ~ "126848876",  # rs902126
      as.character(pos_GRCh37) %in% c("127901649") ~ "126889404",  # rs6984837
      as.character(pos_GRCh37) %in% c("127907398") ~ "126895153",  # rs35358635
      as.character(pos_GRCh37) %in% c("127922200") ~ "126909955",  # rs7011138
      as.character(pos_GRCh37) %in% c("128027954") ~ "127015709",  # rs7463326
      as.character(pos_GRCh37) %in% c("127997652") ~ "126985407",  # rs201495744
      as.character(pos_GRCh37) %in% c("127998576") ~ "126986331",  # rs72725817
      as.character(pos_GRCh37) %in% c("128058136") ~ "127045891",  # rs180753027
      as.character(pos_GRCh37) %in% c("128074815") ~ "127062570",  # rs72725854
      as.character(pos_GRCh37) %in% c("128077146") ~ "127064901",  # rs77541621
      as.character(pos_GRCh37) %in% c("128103969") ~ "127091724",  # rs72725879
      as.character(pos_GRCh37) %in% c("128104117") ~ "127091872",  # rs183373024
      as.character(pos_GRCh37) %in% c("128104218") ~ "127091973",  # rs78809737
      as.character(pos_GRCh37) %in% c("128107192") ~ "127094947",  # rs60681470
      as.character(pos_GRCh37) %in% c("128215636") ~ "127203391",  # rs182395051
      as.character(pos_GRCh37) %in% c("128325355") ~ "127313110",  # rs201057014
      as.character(pos_GRCh37) %in% c("128342866") ~ "127330621",  # rs17464492
      as.character(pos_GRCh37) %in% c("128413305") ~ "127401060",  # rs6983267
      as.character(pos_GRCh37) %in% c("128532137") ~ "127519892",  # rs10090154
      as.character(pos_GRCh37) %in% c("128531689") ~ "127519444",  # rs11986220
      as.character(pos_GRCh37) %in% c("128535543") ~ "127523298",  # rs34265760
      as.character(pos_GRCh37) %in% c("128540776") ~ "127528531",  # rs12549761
      as.character(pos_GRCh37) %in% c("128568550") ~ "127556305",  # rs4285449
      as.character(pos_GRCh37) %in% c("3236083") ~ "3236083",  # rs72697078
      as.character(pos_GRCh37) %in% c("14145058") ~ "14145059",  # rs7045411
      as.character(pos_GRCh37) %in% c("18554773") ~ "18554775",  # rs34540271
      as.character(pos_GRCh37) %in% c("18564470") ~ "18564472",  # rs12004058
      as.character(pos_GRCh37) %in% c("19072246") ~ "19072248",  # rs10122990
      as.character(pos_GRCh37) %in% c("19061659") ~ "19061661",  # rs2171954
      as.character(pos_GRCh37) %in% c("22041998") ~ "22041999",  # rs17694493
      as.character(pos_GRCh37) %in% c("34049779") ~ "34049781",  # rs10122495
      as.character(pos_GRCh37) %in% c("33931398") ~ "33931400",  # rs307645
      as.character(pos_GRCh37) %in% c("82090723") ~ "79475808",  # rs142727307
      as.character(pos_GRCh37) %in% c("109532734") ~ "106770453",  # rs4451364
      as.character(pos_GRCh37) %in% c("110144887") ~ "107382606",  # rs817872
      as.character(pos_GRCh37) %in% c("110147502") ~ "107385221",  # rs12156453
      as.character(pos_GRCh37) %in% c("110290217") ~ "107527936",  # rs143655302
      as.character(pos_GRCh37) %in% c("110714963") ~ "107952682",  # rs7034326
      as.character(pos_GRCh37) %in% c("112969744") ~ "110207464",  # rs201081105
      as.character(pos_GRCh37) %in% c("127044135") ~ "124281856",  # rs7861040
      as.character(pos_GRCh37) %in% c("130430116") ~ "127667837",  # rs2241167
      as.character(pos_GRCh37) %in% c("130376248") ~ "127613969",  # rs6478788
      as.character(pos_GRCh37) %in% c("132573536") ~ "129811257",  # rs12634
      as.character(pos_GRCh37) %in% c("838636") ~ "792696",  # rs12781100
      as.character(pos_GRCh37) %in% c("897201") ~ "851261",  # rs34487581
      as.character(pos_GRCh37) %in% c("5726821") ~ "5684858",  # rs11259192
      as.character(pos_GRCh37) %in% c("5865257") ~ "5823294",  # rs72772400
      as.character(pos_GRCh37) %in% c("28147582") ~ "27858653",  # rs56137930
      as.character(pos_GRCh37) %in% c("46104943") ~ "45609495",  # rs7075427
      as.character(pos_GRCh37) %in% c("47599029") ~ "46227793",  # rs11599847
      as.character(pos_GRCh37) %in% c("51549496") ~ "46046326",  # rs10993994
      as.character(pos_GRCh37) %in% c("80236999") ~ "78477242",  # rs11817544
      as.character(pos_GRCh37) %in% c("80238015") ~ "78478258",  # rs59483558
      as.character(pos_GRCh37) %in% c("80835998") ~ "79076241",  # rs12412705
      as.character(pos_GRCh37) %in% c("90195149") ~ "88435392",  # rs1935581
      as.character(pos_GRCh37) %in% c("90167065") ~ "88407308",  # rs792207
      as.character(pos_GRCh37) %in% c("104428716") ~ "102668959",  # rs12262998
      as.character(pos_GRCh37) %in% c("104422046") ~ "102662289",  # rs4522108
      as.character(pos_GRCh37) %in% c("105682296") ~ "103922538",  # rs11813268
      as.character(pos_GRCh37) %in% c("114711755") ~ "112951996",  # rs10885396
      as.character(pos_GRCh37) %in% c("114715598") ~ "112955839",  # rs2104598
      as.character(pos_GRCh37) %in% c("115078694") ~ "113318935",  # rs528488
      as.character(pos_GRCh37) %in% c("115853630") ~ "114093871",  # rs677062
      as.character(pos_GRCh37) %in% c("122700410") ~ "120940897",  # rs10886810
      as.character(pos_GRCh37) %in% c("122794926") ~ "121035413",  # rs4558107
      as.character(pos_GRCh37) %in% c("122834482") ~ "121074969",  # rs140783917
      as.character(pos_GRCh37) %in% c("122836430") ~ "121076917",  # rs77612511
      as.character(pos_GRCh37) %in% c("123054018") ~ "121294504",  # rs10788167
      as.character(pos_GRCh37) %in% c("123048008") ~ "121288494",  # rs10886900
      as.character(pos_GRCh37) %in% c("123185303") ~ "121425789",  # rs10749415
      as.character(pos_GRCh37) %in% c("123341525") ~ "121582011",  # rs34032268
      as.character(pos_GRCh37) %in% c("126697494") ~ "125008925",  # rs12769682
      as.character(pos_GRCh37) %in% c("126693030") ~ "125004461",  # rs11371876
      as.character(pos_GRCh37) %in% c("1507512") ~ "1486282",  # rs1881502
      as.character(pos_GRCh37) %in% c("2234093") ~ "2212863",  # rs11043143
      as.character(pos_GRCh37) %in% c("2232344") ~ "2211114",  # rs7123299
      as.character(pos_GRCh37) %in% c("3160232") ~ "3139002",  # rs57372985
      as.character(pos_GRCh37) %in% c("7547587") ~ "7526356",  # rs61890184
      as.character(pos_GRCh37) %in% c("9076556") ~ "9055009",  # rs117833876
      as.character(pos_GRCh37) %in% c("32996805") ~ "32975259",  # rs11032076
      as.character(pos_GRCh37) %in% c("34621563") ~ "34600016",  # rs74897609
      as.character(pos_GRCh37) %in% c("47428209") ~ "47406658",  # rs68010938
      as.character(pos_GRCh37) %in% c("47422861") ~ "47401310",  # rs10838703
      as.character(pos_GRCh37) %in% c("58902679") ~ "59135206",  # rs1048374
      as.character(pos_GRCh37) %in% c("58925860") ~ "59158387",  # rs17153415
      as.character(pos_GRCh37) %in% c("58919922") ~ "59152449",  # rs533676902
      as.character(pos_GRCh37) %in% c("61908440") ~ "62140968",  # rs2277283
      as.character(pos_GRCh37) %in% c("66842469") ~ "67074998",  # rs11227678
      as.character(pos_GRCh37) %in% c("66951965") ~ "67184494",  # rs12785905
      as.character(pos_GRCh37) %in% c("66897377") ~ "67129906",  # rs35153256
      as.character(pos_GRCh37) %in% c("68882926") ~ "69115458",  # rs3018690
      as.character(pos_GRCh37) %in% c("68980788") ~ "69213321",  # rs11825796
      as.character(pos_GRCh37) %in% c("69002342") ~ "69234875",  # rs11228580
      as.character(pos_GRCh37) %in% c("69463273") ~ "69648505",  # rs3918298
      as.character(pos_GRCh37) %in% c("69453985") ~ "69639217",  # rs36225067
      as.character(pos_GRCh37) %in% c("76267331") ~ "76556287",  # rs56159348
      as.character(pos_GRCh37) %in% c("76208656") ~ "76497612",  # rs7928373
      as.character(pos_GRCh37) %in% c("77671281") ~ "77960235",  # rs12796422
      as.character(pos_GRCh37) %in% c("102401661") ~ "102530930",  # rs11568818
      as.character(pos_GRCh37) %in% c("102440927") ~ "102570196",  # rs7945830
      as.character(pos_GRCh37) %in% c("108354874") ~ "108484147",  # rs10749918
      as.character(pos_GRCh37) %in% c("108468768") ~ "108598041",  # rs9666351
      as.character(pos_GRCh37) %in% c("108357137") ~ "108486410",  # rs74911261
      as.character(pos_GRCh37) %in% c("111587901") ~ "111717177",  # rs4936662
      as.character(pos_GRCh37) %in% c("113700546") ~ "113829824",  # rs5794883
      as.character(pos_GRCh37) %in% c("125054793") ~ "125184897",  # rs138466039
      as.character(pos_GRCh37) %in% c("129328974") ~ "129459079",  # rs79186742
      as.character(pos_GRCh37) %in% c("134266372") ~ "134396478",  # rs878987
      as.character(pos_GRCh37) %in% c("134280503") ~ "134410609",  # rs28378882
      as.character(pos_GRCh37) %in% c("12871099") ~ "12718165",  # rs2066827
      as.character(pos_GRCh37) %in% c("12877983") ~ "12725049",  # rs77216612
      as.character(pos_GRCh37) %in% c("12879254") ~ "12726320",  # rs7956514
      as.character(pos_GRCh37) %in% c("14416918") ~ "14263984",  # rs10845938
      as.character(pos_GRCh37) %in% c("14421112") ~ "14268178",  # rs11055891
      as.character(pos_GRCh37) %in% c("14892925") ~ "14739991",  # rs4764122
      as.character(pos_GRCh37) %in% c("14933522") ~ "14780588",  # rs7307567
      as.character(pos_GRCh37) %in% c("48419618") ~ "48025835",  # rs80130819
      as.character(pos_GRCh37) %in% c("49672714") ~ "49278931",  # rs56222401
      as.character(pos_GRCh37) %in% c("53308932") ~ "52915148",  # rs113925811
      as.character(pos_GRCh37) %in% c("53312612") ~ "52918828",  # rs73110464
      as.character(pos_GRCh37) %in% c("53329231") ~ "52935447",  # rs187809440
      as.character(pos_GRCh37) %in% c("53315665") ~ "52921881",  # rs142025399
      as.character(pos_GRCh37) %in% c("65012824") ~ "64619044",  # rs7968403
      as.character(pos_GRCh37) %in% c("66406903") ~ "66013123",  # rs975917
      as.character(pos_GRCh37) %in% c("90156377") ~ "89762600",  # rs4842687
      as.character(pos_GRCh37) %in% c("90167191") ~ "89773414",  # rs2037376
      as.character(pos_GRCh37) %in% c("102446675") ~ "102052897",  # rs77121786
      as.character(pos_GRCh37) %in% c("102541667") ~ "102147889",  # rs76825149
      as.character(pos_GRCh37) %in% c("109994870") ~ "109557065",  # rs9593
      as.character(pos_GRCh37) %in% c("114685571") ~ "114247766",  # rs1270884
      as.character(pos_GRCh37) %in% c("114666099") ~ "114228294",  # rs2555009
      as.character(pos_GRCh37) %in% c("115094260") ~ "114656455",  # rs11067228
      as.character(pos_GRCh37) %in% c("133067989") ~ "132491403",  # rs7295014
      as.character(pos_GRCh37) %in% c("36272524") ~ "35698387",  # rs275960
      as.character(pos_GRCh37) %in% c("51076440") ~ "50502304",  # rs1327653
      as.character(pos_GRCh37) %in% c("51065295") ~ "50491159",  # rs35679553
      as.character(pos_GRCh37) %in% c("51088050") ~ "50513914",  # rs12583030
      as.character(pos_GRCh37) %in% c("51471022") ~ "50896886",  # rs9563009
      as.character(pos_GRCh37) %in% c("73716861") ~ "73142724",  # rs7489409
      as.character(pos_GRCh37) %in% c("73995877") ~ "73421740",  # rs7336001
      as.character(pos_GRCh37) %in% c("76227427") ~ "75653291",  # rs7995740
      as.character(pos_GRCh37) %in% c("92013944") ~ "91361690",  # rs9523312
      as.character(pos_GRCh37) %in% c("109574129") ~ "108921781",  # rs9583309
      as.character(pos_GRCh37) %in% c("110112568") ~ "109460221",  # rs12873308
      as.character(pos_GRCh37) %in% c("110360784") ~ "109708437",  # rs75823044
      as.character(pos_GRCh37) %in% c("23305649") ~ "22836440",  # rs1004030
      as.character(pos_GRCh37) %in% c("23785808") ~ "23316599",  # rs11333310
      as.character(pos_GRCh37) %in% c("24691151") ~ "24221945",  # rs117531891
      as.character(pos_GRCh37) %in% c("37136194") ~ "36666989",  # rs6571758
      as.character(pos_GRCh37) %in% c("37102674") ~ "36633469",  # rs12885529
      as.character(pos_GRCh37) %in% c("37634568") ~ "37165363",  # rs145024784
      as.character(pos_GRCh37) %in% c("37931822") ~ "37462617",  # rs146403404
      as.character(pos_GRCh37) %in% c("38144592") ~ "37675387",  # rs11849126
      as.character(pos_GRCh37) %in% c("53387109") ~ "52920391",  # rs4901313
      as.character(pos_GRCh37) %in% c("53333116") ~ "52866398",  # rs62003517
      as.character(pos_GRCh37) %in% c("61106699") ~ "60639981",  # rs8005621
      as.character(pos_GRCh37) %in% c("61157015") ~ "60690297",  # rs2351179
      as.character(pos_GRCh37) %in% c("61724753") ~ "61258035",  # rs11158335
      as.character(pos_GRCh37) %in% c("64606132") ~ "64139414",  # rs114053368
      as.character(pos_GRCh37) %in% c("64687926") ~ "64221208",  # rs79133931
      as.character(pos_GRCh37) %in% c("68923908") ~ "68457191",  # rs2093202
      as.character(pos_GRCh37) %in% c("68989808") ~ "68523091",  # rs1023529
      as.character(pos_GRCh37) %in% c("69134264") ~ "68667547",  # rs767127
      as.character(pos_GRCh37) %in% c("70756333") ~ "70289616",  # rs17565772
      as.character(pos_GRCh37) %in% c("70833819") ~ "70367102",  # rs1044527
      as.character(pos_GRCh37) %in% c("75378185") ~ "74911482",  # rs3213716
      as.character(pos_GRCh37) %in% c("91751808") ~ "91285464",  # rs17802308
      as.character(pos_GRCh37) %in% c("93464018") ~ "92997673",  # rs12586382
      as.character(pos_GRCh37) %in% c("94844947") ~ "94378610",  # rs28929474
      as.character(pos_GRCh37) %in% c("95104745") ~ "94638408",  # rs72695000
      as.character(pos_GRCh37) %in% c("103192755") ~ "102726418",  # rs201750321
      as.character(pos_GRCh37) %in% c("104203938") ~ "103737601",  # rs8004408
      as.character(pos_GRCh37) %in% c("105268228") ~ "104801891",  # rs4983387
      as.character(pos_GRCh37) %in% c("25621639") ~ "25376492",  # rs757438
      as.character(pos_GRCh37) %in% c("33364382") ~ "33072181",  # rs922511
      as.character(pos_GRCh37) %in% c("40965044") ~ "40672846",  # rs11561564
      as.character(pos_GRCh37) %in% c("40939652") ~ "40647454",  # rs8041443
      as.character(pos_GRCh37) %in% c("56022928") ~ "55730730",  # rs12912118
      as.character(pos_GRCh37) %in% c("56385868") ~ "56093670",  # rs33984059
      as.character(pos_GRCh37) %in% c("59954336") ~ "59662137",  # rs6151589
      as.character(pos_GRCh37) %in% c("66835704") ~ "66543366",  # rs74634457
      as.character(pos_GRCh37) %in% c("66833344") ~ "66541006",  # rs112914414
      as.character(pos_GRCh37) %in% c("66942093") ~ "66649755",  # rs8023793
      as.character(pos_GRCh37) %in% c("70668824") ~ "70376485",  # rs12913603
      as.character(pos_GRCh37) %in% c("70654943") ~ "70362604",  # rs34627158
      as.character(pos_GRCh37) %in% c("91505779") ~ "90962549",  # rs79548680
      as.character(pos_GRCh37) %in% c("99200467") ~ "98657238",  # rs28680698
      as.character(pos_GRCh37) %in% c("101767591") ~ "101227386",  # rs2412005
      as.character(pos_GRCh37) %in% c("1663267") ~ "1613266",  # rs75854896
      as.character(pos_GRCh37) %in% c("4311285") ~ "4261284",  # rs111832163
      as.character(pos_GRCh37) %in% c("4420817") ~ "4370816",  # rs35193388
      as.character(pos_GRCh37) %in% c("4940994") ~ "4890993",  # rs760118
      as.character(pos_GRCh37) %in% c("54320197") ~ "54286285",  # rs3751723
      as.character(pos_GRCh37) %in% c("54469331") ~ "54435419",  # rs7188897
      as.character(pos_GRCh37) %in% c("54473072") ~ "54439160",  # rs1420285
      as.character(pos_GRCh37) %in% c("54678305") ~ "54644393",  # rs13380763
      as.character(pos_GRCh37) %in% c("54683802") ~ "54649890",  # rs3893264
      as.character(pos_GRCh37) %in% c("57654576") ~ "57620664",  # rs11863709
      as.character(pos_GRCh37) %in% c("57692456") ~ "57658544",  # rs11866053
      as.character(pos_GRCh37) %in% c("57700469") ~ "57666557",  # rs8045257
      as.character(pos_GRCh37) %in% c("67331854") ~ "67297951",  # rs9930024
      as.character(pos_GRCh37) %in% c("73024276") ~ "72990377",  # rs1858800
      as.character(pos_GRCh37) %in% c("79847632") ~ "79813735",  # rs28709974
      as.character(pos_GRCh37) %in% c("79929808") ~ "79895911",  # rs12445706
      as.character(pos_GRCh37) %in% c("82166181") ~ "82132576",  # rs8052913
      as.character(pos_GRCh37) %in% c("82169833") ~ "82136228",  # rs7405197
      as.character(pos_GRCh37) %in% c("85005745") ~ "84972139",  # rs72801582
      as.character(pos_GRCh37) %in% c("618965") ~ "715725",  # rs684232
      as.character(pos_GRCh37) %in% c("1782952") ~ "1879658",  # rs5030755
      as.character(pos_GRCh37) %in% c("7571752") ~ "7668434",  # rs78378222
      as.character(pos_GRCh37) %in% c("7790138") ~ "7886820",  # rs78044421
      as.character(pos_GRCh37) %in% c("7801082") ~ "7897764",  # rs9895704
      as.character(pos_GRCh37) %in% c("7803118") ~ "7899800",  # rs28441558
      as.character(pos_GRCh37) %in% c("7810596") ~ "7907278",  # rs60424695
      as.character(pos_GRCh37) %in% c("12585459") ~ "12682142",  # rs72811270
      as.character(pos_GRCh37) %in% c("17857626") ~ "17954312",  # rs5819638
      as.character(pos_GRCh37) %in% c("29893888") ~ "31566869",  # rs73991216
      as.character(pos_GRCh37) %in% c("30092898") ~ "31765879",  # rs4795646
      as.character(pos_GRCh37) %in% c("30096784") ~ "31769765",  # rs34441237
      as.character(pos_GRCh37) %in% c("36047417") ~ "37687414",  # rs3110641
      as.character(pos_GRCh37) %in% c("36074979") ~ "37714971",  # rs11649743
      as.character(pos_GRCh37) %in% c("36103565") ~ "37743574",  # rs11263763
      as.character(pos_GRCh37) %in% c("42515327") ~ "44437959",  # rs67633759
      as.character(pos_GRCh37) %in% c("46805705") ~ "48728343",  # rs138213197
      as.character(pos_GRCh37) %in% c("47380305") ~ "49302943",  # rs2960158
      as.character(pos_GRCh37) %in% c("47453937") ~ "49376575",  # rs7212344
      as.character(pos_GRCh37) %in% c("47398245") ~ "49320883",  # rs565189650
      as.character(pos_GRCh37) %in% c("47405338") ~ "49327976",  # rs73324308
      as.character(pos_GRCh37) %in% c("56426027") ~ "58348666",  # rs12938538
      as.character(pos_GRCh37) %in% c("56487636") ~ "58410275",  # rs2526370
      as.character(pos_GRCh37) %in% c("61109624") ~ "63032263",  # rs72843168
      as.character(pos_GRCh37) %in% c("66029037") ~ "68032921",  # rs199554208
      as.character(pos_GRCh37) %in% c("69117532") ~ "71121391",  # rs148511027
      as.character(pos_GRCh37) %in% c("69108127") ~ "71111986",  # rs984434
      as.character(pos_GRCh37) %in% c("20015517") ~ "22435554",  # rs12607091
      as.character(pos_GRCh37) %in% c("45381011") ~ "47854640",  # rs140293195
      as.character(pos_GRCh37) %in% c("45522194") ~ "47995823",  # rs11660291
      as.character(pos_GRCh37) %in% c("51771322") ~ "54244952",  # rs8089411
      as.character(pos_GRCh37) %in% c("51830294") ~ "54303924",  # rs12955457
      as.character(pos_GRCh37) %in% c("56745999") ~ "59078767",  # rs35283980
      as.character(pos_GRCh37) %in% c("56746315") ~ "59079083",  # rs12956892
      as.character(pos_GRCh37) %in% c("60961193") ~ "63293960",  # rs11381388
      as.character(pos_GRCh37) %in% c("60988146") ~ "63320913",  # rs62098660
      as.character(pos_GRCh37) %in% c("73035513") ~ "75323558",  # rs11876000
      as.character(pos_GRCh37) %in% c("73033372") ~ "75321417",  # rs6566073
      as.character(pos_GRCh37) %in% c("74536905") ~ "76824949",  # rs11663079
      as.character(pos_GRCh37) %in% c("76770820") ~ "79010820",  # rs9959454
      as.character(pos_GRCh37) %in% c("76768773") ~ "79008773",  # rs7236307
      as.character(pos_GRCh37) %in% c("76715227") ~ "78955227",  # rs8096321
      as.character(pos_GRCh37) %in% c("17228554") ~ "17117744",  # rs10412482
      as.character(pos_GRCh37) %in% c("17225128") ~ "17114318",  # rs10421727
      as.character(pos_GRCh37) %in% c("19657632") ~ "19546823",  # rs7245983
      as.character(pos_GRCh37) %in% c("32168343") ~ "31677437",  # rs17501397
      as.character(pos_GRCh37) %in% c("38442897") ~ "37952257",  # rs353411
      as.character(pos_GRCh37) %in% c("38548094") ~ "38057454",  # rs59710626
      as.character(pos_GRCh37) %in% c("38738130") ~ "38247490",  # rs4802297
      as.character(pos_GRCh37) %in% c("38743962") ~ "38253322",  # rs12976534
      as.character(pos_GRCh37) %in% c("41985931") ~ "41480023",  # rs11673591
      as.character(pos_GRCh37) %in% c("50082587") ~ "49579330",  # rs67546213
      as.character(pos_GRCh37) %in% c("51345568") ~ "50842312",  # rs2659051
      as.character(pos_GRCh37) %in% c("51354397") ~ "50851141",  # rs11665748
      as.character(pos_GRCh37) %in% c("51361382") ~ "50858126",  # rs61752561
      as.character(pos_GRCh37) %in% c("51362715") ~ "50859459",  # rs76765083
      as.character(pos_GRCh37) %in% c("51360840") ~ "50857584",  # rs62113212
      as.character(pos_GRCh37) %in% c("867324") ~ "886681",  # rs6039055
      as.character(pos_GRCh37) %in% c("25216113") ~ "25235477",  # rs11698745
      as.character(pos_GRCh37) %in% c("30297133") ~ "31709330",  # rs6058431
      as.character(pos_GRCh37) %in% c("31347512") ~ "32759706",  # rs11480453
      as.character(pos_GRCh37) %in% c("31352927") ~ "32765121",  # rs2424905
      as.character(pos_GRCh37) %in% c("32358788") ~ "33770982",  # rs34633531
      as.character(pos_GRCh37) %in% c("34006970") ~ "35419167",  # rs6141551
      as.character(pos_GRCh37) %in% c("34025983") ~ "35438203",  # rs143383
      as.character(pos_GRCh37) %in% c("49548807") ~ "50932270",  # rs73909841
      as.character(pos_GRCh37) %in% c("49564211") ~ "50947674",  # rs76685994
      as.character(pos_GRCh37) %in% c("52186837") ~ "53570298",  # rs61744628
      as.character(pos_GRCh37) %in% c("52464719") ~ "53848180",  # rs6126986
      as.character(pos_GRCh37) %in% c("52463408") ~ "53846869",  # rs4809941
      as.character(pos_GRCh37) %in% c("55578300") ~ "57003244",  # rs1276400
      as.character(pos_GRCh37) %in% c("61001061") ~ "62426005",  # rs11467114
      as.character(pos_GRCh37) %in% c("60999737") ~ "62424681",  # rs35418657
      as.character(pos_GRCh37) %in% c("62229989") ~ "63598636",  # rs381331
      as.character(pos_GRCh37) %in% c("62263538") ~ "63632185",  # rs74179816
      as.character(pos_GRCh37) %in% c("62307517") ~ "63676164",  # rs3787099
      as.character(pos_GRCh37) %in% c("62311129") ~ "63679776",  # rs77552606
      as.character(pos_GRCh37) %in% c("62327199") ~ "63695846",  # rs1056990
      as.character(pos_GRCh37) %in% c("62374389") ~ "63743036",  # rs1058319
      as.character(pos_GRCh37) %in% c("62441171") ~ "63809818",  # rs150947563
      as.character(pos_GRCh37) %in% c("18144042") ~ "16771723",  # rs2824039
      as.character(pos_GRCh37) %in% c("35535103") ~ "34162803",  # rs10154043
      as.character(pos_GRCh37) %in% c("36061995") ~ "34689696",  # rs7275340
      as.character(pos_GRCh37) %in% c("37492881") ~ "36120583",  # rs2242798
      as.character(pos_GRCh37) %in% c("38081480") ~ "36709180",  # rs116988298
      as.character(pos_GRCh37) %in% c("40296411") ~ "38924487",  # rs11701433
      as.character(pos_GRCh37) %in% c("40289167") ~ "38917243",  # rs2836750
      as.character(pos_GRCh37) %in% c("42866332") ~ "41494405",  # rs61735792
      as.character(pos_GRCh37) %in% c("42882462") ~ "41510535",  # rs9978557
      as.character(pos_GRCh37) %in% c("42884031") ~ "41512104",  # rs9984523
      as.character(pos_GRCh37) %in% c("19749525") ~ "19762002",  # rs1978060
      as.character(pos_GRCh37) %in% c("28888939") ~ "28492951",  # rs9625483
      as.character(pos_GRCh37) %in% c("28990275") ~ "28594287",  # rs12165913
      as.character(pos_GRCh37) %in% c("29091147") ~ "28695159",  # rs17886163
      as.character(pos_GRCh37) %in% c("29091856") ~ "28695868",  # rs555607708
      as.character(pos_GRCh37) %in% c("30456153") ~ "30060164",  # rs193478
      as.character(pos_GRCh37) %in% c("39138332") ~ "38742327",  # rs138708
      as.character(pos_GRCh37) %in% c("40499107") ~ "40103103",  # rs34584683
      as.character(pos_GRCh37) %in% c("43499741") ~ "43103735",  # rs6003062
      as.character(pos_GRCh37) %in% c("43500212") ~ "43104206",  # rs5759167
      as.character(pos_GRCh37) %in% c("45698149") ~ "45302268",  # rs9615099
      as.character(pos_GRCh37) %in% c("45728370") ~ "45332489",  # rs6007594
      as.character(pos_GRCh37) %in% c("9811095") ~ "9843055",  # rs960417
      as.character(pos_GRCh37) %in% c("11482634") ~ "11464514",  # rs17321482
      as.character(pos_GRCh37) %in% c("11462074") ~ "11443954",  # rs61018477
      as.character(pos_GRCh37) %in% c("16878663") ~ "16860540",  # rs5924562
      as.character(pos_GRCh37) %in% c("30896320") ~ "30878203",  # rs5972255
      as.character(pos_GRCh37) %in% c("49539082") ~ "49774479",  # rs4824810
      as.character(pos_GRCh37) %in% c("51245276") ~ "51502424",  # rs11338635
      as.character(pos_GRCh37) %in% c("52695895") ~ "52666845",  # rs5943724
      as.character(pos_GRCh37) %in% c("54454406") ~ "54427973",  # rs4826594
      as.character(pos_GRCh37) %in% c("66825357") ~ "67605515",  # rs5919393
      as.character(pos_GRCh37) %in% c("66751555") ~ "67531713",  # rs7888856
      as.character(pos_GRCh37) %in% c("67092145") ~ "67872303",  # rs5965467
      as.character(pos_GRCh37) %in% c("70139908") ~ "70920058",  # rs371707439
      as.character(pos_GRCh37) %in% c("70108889") ~ "70889039",  # rs13441059
      as.character(pos_GRCh37) %in% c("70432708") ~ "71212858",  # rs747181
      as.character(pos_GRCh37) %in% c("124586414") ~ "125452565",  # rs5931727
      TRUE ~ NA_character_
    )
   ) %>% dplyr::mutate(pos = pos_GRCh38) # Set the pos column to the new GRCh38 positions

# Write csv to visualise

write.csv(raw_data_h38, "raw_data_h38.csv", row.names = FALSE)

In [ ]:
## This code block creates the beta (effect size) column by choosing
## an odds ratio column, then calculates beta as the natural log of the odds ratio. 

if (source == "Wang" & weightings == "Multiethnic") {
    OR_column <- "OR_Multiethnic_Marginal"
} else if (source == "Wang" & weightings == "EUR") {
    OR_column <- "OR_EUR"
} else if (source == "Wang" & weightings == "AFR") {
    OR_column <- "OR_AFR"
} else if (source == "Wang" & weightings == "EAS") {
    OR_column <- "OR_EAS"
} else if (source == "Wang" & weightings == "HIS") {
    OR_column <- "OR_HIS"
} else if (source == "Conti" & weightings == "Multiethnic") {
    OR_column <- "OR_Multiethnic"
} else if (source == "Conti" & weightings == "EUR") {
    OR_column <- "OR_EUR"
} else if (source == "Conti" & weightings == "AFR") {
    OR_column <- "OR_AFR"
} else if (source == "Conti" & weightings == "EAS") {
    OR_column <- "OR_EAS"
} else if (source == "Conti" & weightings == "HIS") {
    OR_column <- "OR_HIS"
} else {
    stop("Invalid source or weightings combination. Please check your inputs.")
}

## Ensure all OR columns are numeric and calculate beta from OR, storing dataframe as "pre_varlist"

if (source == "Wang") { 
pre_varlist <- raw_data_h38 %>%
    dplyr::filter(Wang == "x") %>% ### Filter to only SNPs in the Wang 451 score
    dplyr::mutate(dplyr::across(dplyr::all_of(c("OR_Multiethnic_Marginal", "OR_EUR", "OR_AFR", "OR_EAS", "OR_HIS")), as.numeric)) %>% ### Ensure numeric
    dplyr::mutate(beta = log(.data[[OR_column]])) ### Calculate beta from selected OR column
} else if (source == "Conti") {
    pre_varlist <- raw_data_h38 %>%
    dplyr::mutate(dplyr::across(dplyr::all_of(c("OR_Multiethnic", "OR_EUR", "OR_AFR", "OR_EAS", "OR_HIS")), as.numeric)) %>% 
    dplyr::mutate(beta = log(.data[[OR_column]])) 
} else {
    stop("Invalid source. Please check your input.")
}

## Subset columns for varlist and remove any rows with NA values for the beta column

varlist <- pre_varlist %>%
    dplyr::select(chr, pos, effect_allele, other_allele, beta) %>%
    dplyr::filter(!is.na(beta)) 

## Write varlist to tsv ready for upload

write.table(varlist, paste0(source, weightings, "_varlist.tsv"), row.names = FALSE, col.names = TRUE, quote = FALSE, sep = "\t")

# Upload to project space - special command used here to upload directly to a specific folder in a specific project
## Edit the path and project name as needed. The path should be the same path you intend to have the GRS output files uploaded

system(paste0("bash -c 'unset HTTP_PROXY HTTPS_PROXY http_proxy https_proxy; dx upload ", source, weightings, "_varlist.tsv --path ", output_dir, source, "/", source, weightings, "_varlist.tsv'"))

In [ ]:
## Run GRS calculation script

system(paste('bash polygenic_score.sh "', source, weightings, '" "', output_dir, '"', sep = ""))